## Q1. Personalized Knowledge Base

In [13]:
import pandas as pd

roll_no = "1024170057"

last_two = roll_no[-2:]
print("Last two digits:", last_two)

categories = ["billing", "account", "general"]

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi",
        "category": "billing"
    }
]

personalized_entries = []

for digit in last_two:
    d = int(digit)
    category = categories[d % 3]

    if category == "general":
        question = "where can i find general information"
        answer = "General information is available in the help section."
        keywords = "information help details"

    elif category == "account":
        question = "how do i update my mobile number"
        answer = "You can update your mobile number from your account settings."
        keywords = "mobile number update"

    else:
        question = "how can i check my payment"
        answer = "You can check your payment from the payments section."
        keywords = "payment check transaction"

    personalized_entries.append({
        "question": question,
        "answer": answer,
        "keywords": keywords,
        "category": category
    })

faq_data = fixed_entries + personalized_entries

df = pd.DataFrame(faq_data)

print(df)

Last two digits: 57
                               question  \
0                what is the annual fee   
1                 how to reset password   
2           what are your working hours   
3                 how can i pay the fee   
4  where can i find general information   
5      how do i update my mobile number   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  General information is available in the help s...   
5  You can update your mobile number from your ac...   

                   keywords category  
0            fee cost price  billing  
1      password reset login  account  
2         hours timing open  general  
3           pay payment upi  billing  
4  information help details  general  
5      mobile number update  account  


## Q2. Generate and Score a Hypothesis

In [14]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        text = (row["question"] + " " + row["keywords"]).lower()
        text_words = set(text.split())

        matched_words = query_words.intersection(text_words)
        score = len(matched_words)

        if score > 0:
            confidence = score / len(query_words)

            results.append({
                "index": index,
                "question": row["question"],
                "category": row["category"],
                "score": score,
                "confidence": confidence
            })

    result_df = pd.DataFrame(results)

    if len(result_df) > 0:
        result_df = result_df.sort_values(
            by="confidence",
            ascending=False
        )

    return result_df

query = "how can i pay fee"

result = score_query(query, df)

print(result)

   index                              question category  score  confidence
2      3                 how can i pay the fee  billing      5         1.0
4      5      how do i update my mobile number  account      2         0.4
3      4  where can i find general information  general      2         0.4
0      0                what is the annual fee  billing      1         0.2
1      1                 how to reset password  account      1         0.2


## Q3. FAQs from Same Category

In [15]:
def same_category(category_name, df):
    return df[df["category"] == category_name][
        ["question", "category"]
    ]

In [16]:
personalized_category = personalized_entries[0]["category"]

print("Category:", personalized_category)
print(same_category(personalized_category, df))

Category: general
                               question category
2           what are your working hours  general
4  where can i find general information  general


## Q4. Add a New Keyword

In [17]:
entry_number = 0

new_keyword = input("Enter a new keyword: ")

df.loc[entry_number, "keywords"] = (
    df.loc[entry_number, "keywords"] + " " + new_keyword
)

print(df.loc[entry_number])

question       what is the annual fee
answer      The annual fee is Rs 500.
keywords        fee cost price annual
category                      billing
Name: 0, dtype: str


In [18]:
output_file = roll_no + "_faq_data.csv"

df.to_csv(output_file, index=False)

print("File saved as:", output_file)

File saved as: 1024170057_faq_data.csv


In [19]:
saved_df = pd.read_csv(output_file)

print(saved_df)

                               question  \
0                what is the annual fee   
1                 how to reset password   
2           what are your working hours   
3                 how can i pay the fee   
4  where can i find general information   
5      how do i update my mobile number   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  General information is available in the help s...   
5  You can update your mobile number from your ac...   

                   keywords category  
0     fee cost price annual  billing  
1      password reset login  account  
2         hours timing open  general  
3           pay payment upi  billing  
4  information help details  general  
5      mobile number update  account  


## Q5. FAQ Count by Category

In [20]:
category_count = df.groupby("category").size()

print(category_count)

category
account    2
billing    2
general    2
dtype: int64


## Q6. Handling Ties

In [21]:
def score_query_with_tie(query, df):
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        text = (row["question"] + " " + row["keywords"]).lower()
        text_words = set(text.split())

        matched_words = query_words.intersection(text_words)
        score = len(matched_words)

        if score > 0:
            confidence = score / len(query_words)

            results.append({
                "index": index,
                "question": row["question"],
                "category": row["category"],
                "score": score,
                "confidence": confidence
            })

    result_df = pd.DataFrame(results)

    if len(result_df) == 0:
        print("No matching entries found.")
        return result_df

    highest_score = result_df["score"].max()

    best_matches = result_df[
        result_df["score"] == highest_score
    ]

    print("Highest score:", highest_score)
    print()
    print(best_matches)

    return best_matches

In [22]:
tie_query = "fee"

score_query_with_tie(tie_query, df)

Highest score: 1

   index                question category  score  confidence
0      0  what is the annual fee  billing      1         1.0
1      3   how can i pay the fee  billing      1         1.0


,index,question,category,score,confidence
0,0,what is the annual fee,billing,1,1.0
1,3,how can i pay the fee,billing,1,1.0


In [23]:
normal_query = "password"

score_query_with_tie(normal_query, df)

Highest score: 1

   index               question category  score  confidence
0      1  how to reset password  account      1         1.0


,index,question,category,score,confidence
0,1,how to reset password,account,1,1.0
